# I1. The radio sky and its emission

The first notebook of the introductory track ({doc}`../index`), for readers who
know physics but not radio astronomy. It needs no data and runs in a few
seconds.

**Learning goals.** After this notebook you can

- convert between flux density in jansky and power received by a dish;
- tell synchrotron from thermal emission by spectral index;
- convert a flux density into a brightness temperature;
- say why flux standards exist and read one for the calibrator PKS 1934−638;
- predict the thermal noise of one MeerKAT visibility from the radiometer
  equation, and of an image from the number of visibilities.

Every number below is computed in a cell, from rang's flux standards
(`rang.fluxstd`, CASA `setjy`'s models) and from the design of A1, rang's
simulated MeerKAT observation (`rang.simtruth`).

In [ ]:
%matplotlib inline
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rang import fluxstd
from rang import simtruth as st
from rang.notebook import data

K_B = 1.380649e-23  # J/K
C = 299_792_458.0  # m/s
JY = 1e-26  # W m^-2 Hz^-1

## 1. Flux density and the jansky

A radio source is described by its **flux density** S: power per unit area
per unit bandwidth, in jansky (1 Jy = 10⁻²⁶ W m⁻² Hz⁻¹). Radio sources are
faint. A 1 Jy source seen by one MeerKAT dish (13.5 m) in the whole L band
(856-1712 MHz) delivers, in one polarisation (half the power):

In [ ]:
d = st.design("small")
area = math.pi * (13.5 / 2) ** 2
bandwidth = d.fmax - d.fmin
power = 0.5 * 1.0 * JY * area * bandwidth
print(f"dish area {area:.1f} m^2, bandwidth {bandwidth / 1e6:.0f} MHz")
print(f"power from 1 Jy, one polarisation: {power:.2e} W")
print(f"the receiver's own noise power k T Δν for T = 20 K: {K_B * 20 * bandwidth:.2e} W")

The source is about 10⁴ times weaker than the receiver noise. Radio
astronomy is the business of averaging that noise down, which is what the
radiometer equation in section 5 quantifies.

## 2. Spectra: synchrotron and thermal

Most of the radio sky at GHz frequencies is **synchrotron** emission from
relativistic electrons in magnetic fields: a power law S ∝ ν^α with spectral
index α ≈ −0.7 for optically thin emission. **Thermal** (free-free) emission
from ionised gas is S ∝ ν² where it is optically thick and nearly flat
(α ≈ −0.1) where it is thin. A self-absorbed synchrotron source turns over
at low frequency, giving a **peaked spectrum**.

The simulated target field of A1 draws its 40 point sources' spectral
indices from a normal distribution centred on −0.7:

In [ ]:
sky = st.target_sky(d)
alpha = np.array([c["alpha"] for c in sky if c["kind"] == "point"])
print(f"{len(alpha)} point sources: alpha mean {alpha.mean():.2f}, std {alpha.std():.2f}")

nu = np.linspace(0.3e9, 5e9, 400)
tau = (nu / 1.0e9) ** -2.1  # free-free optical depth, 1 at 1 GHz (an assumed example)
spectra = {
    "synchrotron, alpha = -0.7": (nu / 1.284e9) ** -0.7,
    # T_e constant: S ∝ ν² (1 − e^−τ), so ν² where τ ≫ 1 and ν^−0.1 where τ ≪ 1.
    "thermal free-free, tau(1 GHz) = 1": nu**2 * (1 - np.exp(-tau)),
}
fig, ax = plt.subplots(figsize=(7, 3.5))
for label, s in spectra.items():
    ax.loglog(nu / 1e9, s / np.interp(1.284e9, nu, s), label=label)
ax.axvspan(d.fmin / 1e9, d.fmax / 1e9, color="0.9", label="MeerKAT L band")
ax.set(xlabel="frequency (GHz)", ylabel="S / S(1.284 GHz)", title="Spectral shapes")
ax.legend()
plt.show()

The spectral index between two frequencies is α = ln(S₂/S₁) / ln(ν₂/ν₁). In
the free-free curve it runs from about +2 (thick) to −0.1 (thin):

In [ ]:
def spectral_index(s1, s2, nu1, nu2):
    return math.log(s2 / s1) / math.log(nu2 / nu1)


ff = spectra["thermal free-free, tau(1 GHz) = 1"]
for lo, hi in ((0.3e9, 0.4e9), (4e9, 5e9)):
    print(f"free-free alpha between {lo / 1e9:.1f} and {hi / 1e9:.1f} GHz: "
          f"{spectral_index(np.interp(lo, nu, ff), np.interp(hi, nu, ff), lo, hi):+.2f}")

## 3. Brightness temperature

The Rayleigh-Jeans law gives the temperature a black body would need to
produce the observed brightness: T_b = S λ² / (2 k Ω), with Ω the solid angle
of the source. For a Gaussian beam of FWHM θ, Ω = π θ² / (4 ln 2). A
brightness temperature far above 10⁴ K rules out thermal emission from
ionised gas; above about 10¹² K, incoherent synchrotron emission fails too
(the inverse-Compton limit), which is one sign of coherent emission such as a
pulsar's (notebook I9).

In [ ]:
def brightness_temperature(s_jy, freq_hz, fwhm_arcsec):
    theta = fwhm_arcsec / 3600 * math.pi / 180
    omega = math.pi * theta**2 / (4 * math.log(2))
    return s_jy * JY * (C / freq_hz) ** 2 / (2 * K_B * omega)


rows = [
    ("1 mJy in an 8 arcsec beam", 1e-3, 8.0),
    ("1 Jy in an 8 arcsec beam", 1.0, 8.0),
    ("1 Jy in 1 milliarcsecond", 1.0, 1e-3),
]
pd.DataFrame(
    [{"case": c, "T_b at 1.284 GHz (K)": f"{brightness_temperature(s, 1.284e9, th):.3g}"} for c, s, th in rows]
)

## 4. The flux calibrator and flux standards

An interferometer measures correlated voltages in arbitrary units. To put
the image on a jansky scale, an observation includes a **flux calibrator**:
a bright, compact source whose spectrum is known from careful absolute
measurements. MeerKAT's is J1939−6342 (PKS 1934−638), a compact peaked-spectrum
source. Its spectrum is tabulated in **flux standards**, polynomial fits of
log S against log ν. CASA's `setjy` has several; rang reimplements them
(`rang.fluxstd`):

In [ ]:
pd.DataFrame(
    [{"standard": s, "sources": ", ".join(fluxstd.sources(s))} for s in fluxstd.standards()]
)

In [ ]:
freq = d.freq  # A1 small's 64 channel centres
models = {s: fluxstd.flux("1934-638", freq, s)[:, 0] for s in fluxstd.standards() if "1934-638" in fluxstd.sources(s)}
models["Reynolds 1994 (notebook 01)"] = data.reynolds_flux(freq)
fig, ax = plt.subplots(figsize=(7, 3.5))
for label, s in models.items():
    ax.plot(freq / 1e6, s, label=label)
ax.set(xlabel="frequency (MHz)", ylabel="Stokes I (Jy)", title="PKS 1934−638 across MeerKAT L band")
ax.legend()
plt.show()
ref = models["Stevens-Reynolds 2016"]
pd.DataFrame(
    [
        {
            "model": label,
            "S at 1.284 GHz (Jy)": f"{np.interp(1.284e9, freq, s):.3f}",
            "max |difference from Stevens-Reynolds 2016|": f"{np.max(np.abs(s / ref - 1)) * 100:.2f} %",
        }
        for label, s in models.items()
    ]
)

The spectrum peaks inside the band, so its local spectral index changes
sign: a self-absorbed synchrotron source.

In [ ]:
i_mid = len(freq) // 2
local_alpha = np.gradient(np.log(ref), np.log(freq))
print(f"Stevens-Reynolds 2016: alpha {local_alpha[0]:+.2f} at {freq[0] / 1e6:.0f} MHz, "
      f"{local_alpha[i_mid]:+.2f} at {freq[i_mid] / 1e6:.0f} MHz, {local_alpha[-1]:+.2f} at {freq[-1] / 1e6:.0f} MHz")
print(f"peak at {freq[np.argmax(ref)] / 1e6:.0f} MHz (channel resolution {(freq[1] - freq[0]) / 1e6:.1f} MHz)")

Three of the four models are the same polynomial, Reynolds (1994), and agree
to rounding: CASA's Perley-Butler 2010 and Stevens-Reynolds 2016 entries for
this source reuse it. The Baars model is older and differs by the amount the
table shows. The standard a pipeline picks sets the absolute flux scale of
every image calibrated against this source, so always record it. Notebook 01
uses Reynolds (1994) directly; E4's tutorial ({doc}`00-ms-to-image`) and A1
use Stevens-Reynolds 2016.

## 5. The radiometer equation

A receiver of system temperature T_sys on a dish of effective area A_eff
has a **system equivalent flux density** SEFD = 2 k T_sys / A_eff: the flux
density of a source that would double the noise power. Correlating two such
receivers over bandwidth Δν for time Δt gives a complex visibility whose real
and imaginary parts each carry Gaussian noise

σ = SEFD / √(2 Δν Δt).

A1 uses SEFD = 420 Jy, channels of 13.4 MHz and integrations of 8 s:

In [ ]:
dnu = (d.fmax - d.fmin) / d.nchan
sigma = d.sefd / math.sqrt(2 * dnu * d.dt)
print(f"SEFD {d.sefd:.0f} Jy, channel width {dnu / 1e6:.3f} MHz, integration {d.dt:.0f} s")
print(f"sigma per visibility, real and imaginary part: {sigma:.4f} Jy (A1's design records {d.sigma:.4f} Jy)")
print(f"equivalent T_sys/eta for a 13.5 m dish: {d.sefd * JY * area / (2 * K_B):.1f} K")

An image averages many visibilities. With natural weighting and no
flagging, the noise of a Stokes I image, I = (XX + YY)/2, is σ / √2 divided by
the square root of the number of visibilities. For A1 small's target field
(16 antennas, 120 baselines, 64 channels, the target's integrations):

In [ ]:
times, scans, fields, within = st.schedule(d)
target = [i for i, f in enumerate(st.FIELDS) if f[0] == "SIM-TARGET"][0]
nint = int((fields == target).sum())
nant = len(d.antennas)
nvis = nant * (nant - 1) // 2 * d.nchan * nint
sigma_image = sigma / math.sqrt(2) / math.sqrt(nvis)
print(f"{nint} target integrations ({nint * d.dt / 60:.1f} min on source), {nvis:,} visibilities per correlation")
print(f"expected Stokes I image noise (natural weighting, no flags): {sigma_image * 1e6:.0f} µJy/beam")

Notebook I7 compares this number with the residual noise GeMClean reaches on
the same data, where flagging and Briggs weighting raise it.

## Exercises

1. The public fixture used from notebook I2 on has 4,096 channels of 208.984 kHz
   and 8 s integrations. What is σ per visibility for SEFD = 420 Jy?
2. How long must 64 MeerKAT antennas integrate across 800 MHz of usable
   bandwidth to reach 2 µJy/beam in Stokes I, natural weighting?
3. Use `fluxstd.flux("3C286", freq, "Perley-Butler 2010")` to find 3C 286's
   spectral index across L band. Is it thick or thin?
4. A source of 50 mJy is unresolved by a 0.05 arcsec beam at 1.4 GHz. What is
   the lower limit on its brightness temperature, and what does it rule out?

## Where next

- {doc}`i2-visibilities`: what an interferometer measures.
- {doc}`../../concepts/index`: the concepts pages.
- Notebook {doc}`01-data-and-rank` loads real MeerKAT data on PKS 1934−638.